# E · Per-example labels and valid movement re-pairing

This experiment asks whether meaningful movement pairing contributes beyond additional supervision. It crosses coordinate-pretrained, paired-JEPA and direct encoders with per-example measurement labels or valid re-paired change labels.

Read after notebooks 00–06. This notebook uses the prepared bundle and saved evaluation from that same run; the internal recipe group is `L`.


In [ ]:
from pathlib import Path
import json, os, sys

# Find the checkout/release from the notebook's working directory.
ROOT = Path(os.environ.get('GF_ROOT', Path.cwd())).resolve()
while not (ROOT / 'src/gavd6_sjepa').is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert (ROOT / 'src/gavd6_sjepa').is_dir(), 'Open this notebook from the GAVD6 checkout or release.'
sys.path.insert(0, str(ROOT / 'notebooks/gait_fidelity'))
sys.path.insert(0, str(ROOT / 'src'))
from tutorial_helpers import configure, preview_images
study = configure(ROOT)


## Inspect the exact recipe cells

Compare the per-example control and valid re-pairing control with the corresponding paired-change arm from experiment A or C. Re-pairing rearranges training source families while preserving endpoint roles and applicable condition strata, then recomputes the true difference between the new reference endpoints.


In [ ]:
import pandas as pd
plan = study.artifact('plan.json')
group_recipes = [r for r in plan['recipes'] if r['group'] == 'L']
expected_count = 6 if plan.get('experiment_set', 'full') == 'full' else {'M': 4, 'T': 0, 'P': 2, 'I': 4, 'L': 0}['L']
assert len(group_recipes) == expected_count, 'Saved plan differs from the selected experiment set.'
recipe_ids = {r['recipe_id'] for r in group_recipes}
display(pd.DataFrame(group_recipes))
print('Final models:', len(group_recipes) * len(plan['seeds']), 'Seeds:', plan['seeds'])
if not group_recipes:
    print('This group is outside the saved core protocol. Its worked examples are educational; no results are implied.')


## Separate endpoint labels from the relationship between endpoints

Write $a_0,a_1$ for reference right-minus-left knee excursions and
$\hat a_0,\hat a_1$ for restored values. Per-example supervision minimizes
$\{(\hat a_0-a_0)^2+(\hat a_1-a_1)^2\}/(2\cdot180^2)$.
Change supervision minimizes
$\{(\hat a_1-\hat a_0)-(a_1-a_0)\}^2/180^2$.
Both retain coordinate supervision and a short-limb penalty, derived in
notebook 04. Change supervision can cancel a shared endpoint bias, which
is why endpoint accuracy and change accuracy are evaluated separately.

Valid re-pairing keeps baseline endpoints fixed and permutes intervention
endpoints across source families within movement/nuisance strata. Its
reference differences must be calculated again. The example below
reproduces the core permutation and shared-minibatch mechanism on six
**illustrative scalar labels**, not on research predictions.


In [ ]:
import numpy as np
from gavd6_sjepa.research_directions.gait_fidelity.training import paired_batch_cycles, draw_pair_batch
# Rows 0–5 are baseline endpoints, 6–11 their intervention endpoints.
pairs = np.column_stack([np.arange(6), np.arange(6, 12)])
permutation = np.array([1, 0, 3, 2, 5, 4])  # Three closed two-family cycles.
repaired = np.column_stack([pairs[:, 0], pairs[permutation, 1]])
original_labels = np.array([0., 2., 1., 4., 3., 6., 5., 10., 9., 7., 13., 8.])
original_change = original_labels[pairs[:, 1]] - original_labels[pairs[:, 0]]
repaired_change = original_labels[repaired[:, 1]] - original_labels[repaired[:, 0]]
assert not np.array_equal(original_change, repaired_change)
# Traverse the intervention permutation to form complete cycles.
visited, cycles = set(), []
for start in range(len(permutation)):
    if start in visited:
        continue
    current, cycle = start, []
    while current not in visited:
        visited.add(current)
        cycle.append(current)
        current = int(permutation[current])
    assert current == start
    cycles.append(np.asarray(cycle))
for explicit, production in zip(cycles, paired_batch_cycles(pairs, repaired)):
    np.testing.assert_array_equal(explicit, production)
# Complete cycles make the exact endpoint multiset equal at every update.
rng = np.random.default_rng(17)
chosen, size = [], 0
while size < 3:
    cycle = cycles[int(rng.integers(len(cycles)))]
    chosen.append(cycle)
    size += len(cycle)
batch = np.concatenate(chosen)
np.testing.assert_array_equal(batch, draw_pair_batch(cycles, 3, np.random.default_rng(17)))
np.testing.assert_array_equal(np.sort(pairs[batch].ravel()), np.sort(repaired[batch].ravel()))
display(pd.DataFrame({'baseline': pairs[:, 0], 'original_endpoint': pairs[:, 1],
                      'new_endpoint': repaired[:, 1], 'original_change': original_change,
                      'recomputed_change': repaired_change}))
print('Requested pairs: 3; complete-cycle batch pairs:', len(batch))


## Check how the real training control was matched

The training code proposes two-family cycles, with one three-family cycle
for an odd stratum. Across a fixed number of seeded trials, it selects the
proposal minimizing reference-change distribution mismatch. For an equally
sized stratum this is
$W=\operatorname{mean}|\operatorname{sort}(\Delta a')-
\operatorname{sort}(\Delta a)|/180$.
Selection uses training references only. The worst stratum must satisfy the
declared tolerance to support attribution specifically to meaningful pairing.
Pairwise common reference support is recomputed when the new endpoints have
different validity patterns. The displayed toy labels assume full support.


In [ ]:
mismatch = np.mean(abs(np.sort(repaired_change) - np.sort(original_change))) / 180.
print('Illustrative normalized distribution mismatch:', mismatch)
# Inspect retained receipts rather than rerunning the matching search.
ledger = study.artifact('ledger.json')
audits = []
for phase in plan['phases']:
    recipe = phase['recipe']
    if phase['phase'] == 'pretrain' or recipe['group'] != 'L':
        continue
    result = ledger.get('completed', {}).get(phase['phase_id'], {}).get('result', {})
    checkpoint = result.get('checkpoint')
    if checkpoint:
        audit_path = Path(checkpoint).parent / 'pairing-audit.json'
        receipt = json.loads(audit_path.read_text())
        audits.append({'phase_id': phase['phase_id'],
                       **{k: receipt[k] for k in ['endpoint_frequency_preserved',
                            'worst_stratum_mismatch', 'tolerance', 'tolerance_passed']}})
display(pd.DataFrame(audits))


Batch size is nominal because a complete three-cycle may cross its boundary.
All compared objectives use the same cycle sampler, so this overshoot does
not give the re-paired arm extra endpoint exposure. A failed distribution
tolerance stays in the report and limits the pairing claim. It must not be
repaired by searching for a more favorable tolerance after evaluation.


## Follow the shared dependencies

This tutorial inspects the existing central queue. It does not launch a separate copy of its group: that would duplicate pretraining and break the global budget. Notebook 04 launches all groups, and this table identifies the phases that belong to the present comparison.


In [ ]:
final_phases = [p for p in plan['phases'] if p['phase'] != 'pretrain' and p['recipe']['recipe_id'] in recipe_ids]
parent_ids = {parent for p in final_phases for parent in p['depends_on'] if parent != 'prepare'}
selected = [p for p in plan['phases'] if p in final_phases or p['phase_id'] in parent_ids]
display(pd.DataFrame([{'phase_id': p['phase_id'], 'phase': p['phase'], 'seed': p['seed'],
                      'depends_on': ', '.join(p['depends_on'])} for p in selected]))


## Inspect completed checkpoints and learning histories

Each completed phase links its retained checkpoint, history and predictions to its source identity. Missing phases are reported as pending; a checkpoint from another study is not substituted.


In [ ]:
ledger_path = study.work / 'ledger.json'
completed = json.loads(ledger_path.read_text()).get('completed', {}) if ledger_path.exists() else {}
rows = []
for phase in selected:
    saved = completed.get(phase['phase_id'])
    result = saved.get('result', {}) if saved else {}
    rows.append({'phase_id': phase['phase_id'], 'status': 'complete' if saved else 'pending',
                 'checkpoint': result.get('checkpoint'), 'predictions': result.get('predictions')})
display(pd.DataFrame(rows))
history_candidates = []
for row in rows:
    if row['checkpoint']:
        history_path = Path(row['checkpoint']).parent / 'history.json'
        if history_path.exists(): history_candidates.append(history_path)
if history_candidates:
    history_path = history_candidates[0]
    print('First declared completed history:', history_path)
    display(pd.DataFrame(json.loads(history_path.read_text())))
else:
    print('No completed histories yet. Run or resume the central queue from notebook 04.')


## Read group results on the common population

Read each pairing-audit.json before interpreting the comparison. The sampler packs complete two- or three-family permutation cycles into each batch, so all objectives receive the same endpoint multiset at every update. A nominal 16-endpoint batch can contain 18 or 20 endpoints when a three-cycle crosses its boundary; receipts retain the actual counts. Endpoint frequency and reference-change distribution must meet the frozen matching rule. If the declared tolerance cannot be met, report that mismatch and narrow the pairing-specific claim. Reusing the original target differences after re-pairing would create false labels.


In [ ]:
person_path = study.work / 'evaluation/per-person.csv'
if person_path.exists():
    people = pd.read_csv(person_path)
    display(people.loc[people['method'].isin(recipe_ids)])
    coverage_path = study.work / 'evaluation/coverage.csv'
    if coverage_path.exists():
        coverage = pd.read_csv(coverage_path)
        display(coverage.loc[coverage['method'].isin(recipe_ids)])
else:
    print('Evaluation is pending. These recipe cards do not fabricate or extrapolate results.')


Read the matching controls from the other experiment tutorials before attribution. Full evaluation and numerical reconstruction are covered by notebooks 05 and 06.
